In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Birthdate").getOrCreate()
df_people=spark.read.parquet("/databricks-datasets/learning-spark-v2/people/people-10m.parquet/")
df_people.printSchema()

root
 |-- id: integer (nullable = true)
 |-- firstName: string (nullable = true)
 |-- middleName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthDate: timestamp (nullable = true)
 |-- ssn: string (nullable = true)
 |-- salary: integer (nullable = true)



In [0]:
from pyspark.sql.functions import window,desc,col
df_people.orderBy(col("birthDate")).show()


+-------+----------+----------+------------+------+-------------------+-----------+------+
|     id| firstName|middleName|    lastName|gender|          birthDate|        ssn|salary|
+-------+----------+----------+------------+------+-------------------+-----------+------+
|4078354|     Sarah|   Jerilyn|   O'Henecan|     F|1951-12-31 05:00:00|944-53-3373| 86109|
|4640222|   Rebecca| Catharine|       Beden|     F|1951-12-31 05:00:00|929-79-6187| 55846|
|3864759|    Willia|      Nola|     Waggatt|     F|1951-12-31 05:00:00|666-64-5477| 59385|
|4351195|     Maile|  Vasiliki|De Bernardis|     F|1951-12-31 05:00:00|992-97-2492|104510|
|4285095|     Mindy|   Aurelia|   Maggorini|     F|1951-12-31 05:00:00|943-30-7203| 81016|
|4864532|Antoinette|     Arica|  Korneichik|     F|1951-12-31 05:00:00|989-13-5446| 75354|
|  61274|   Basilia|    Janean|    Benedick|     F|1951-12-31 05:00:00|965-20-8148|129242|
|4938761|   Doretta|    Chante|    Danielli|     F|1951-12-31 05:00:00|948-74-4817| 43843|

In [0]:
df_people.groupby("birthDate").count().orderBy(desc("birthDate")).show()

+-------------------+-----+
|          birthDate|count|
+-------------------+-----+
|2000-01-30 05:00:00|  412|
|2000-01-29 05:00:00|  557|
|2000-01-28 05:00:00|  547|
|2000-01-27 05:00:00|  557|
|2000-01-26 05:00:00|  542|
|2000-01-25 05:00:00|  536|
|2000-01-24 05:00:00|  563|
|2000-01-23 05:00:00|  535|
|2000-01-22 05:00:00|  551|
|2000-01-21 05:00:00|  600|
|2000-01-20 05:00:00|  567|
|2000-01-19 05:00:00|  556|
|2000-01-18 05:00:00|  596|
|2000-01-17 05:00:00|  537|
|2000-01-16 05:00:00|  573|
|2000-01-15 05:00:00|  563|
|2000-01-14 05:00:00|  559|
|2000-01-13 05:00:00|  586|
|2000-01-12 05:00:00|  551|
|2000-01-11 05:00:00|  609|
+-------------------+-----+
only showing top 20 rows


In [0]:
df_people_count_datewise=df_people.groupby("birthDate").count().orderBy(desc("birthDate"))
df_people_count_datewise.show()

+-------------------+-----+
|          birthDate|count|
+-------------------+-----+
|2000-01-30 05:00:00|  412|
|2000-01-29 05:00:00|  557|
|2000-01-28 05:00:00|  547|
|2000-01-27 05:00:00|  557|
|2000-01-26 05:00:00|  542|
|2000-01-25 05:00:00|  536|
|2000-01-24 05:00:00|  563|
|2000-01-23 05:00:00|  535|
|2000-01-22 05:00:00|  551|
|2000-01-21 05:00:00|  600|
|2000-01-20 05:00:00|  567|
|2000-01-19 05:00:00|  556|
|2000-01-18 05:00:00|  596|
|2000-01-17 05:00:00|  537|
|2000-01-16 05:00:00|  573|
|2000-01-15 05:00:00|  563|
|2000-01-14 05:00:00|  559|
|2000-01-13 05:00:00|  586|
|2000-01-12 05:00:00|  551|
|2000-01-11 05:00:00|  609|
+-------------------+-----+
only showing top 20 rows


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,lit,count,round,timestamp_add
window_spec=Window.partitionBy("birthDate").orderBy("id")
window_spec_all=Window.partitionBy(col("birthDate"))
df_people.withColumn("rown",row_number().over(window_spec)).withColumn("total_cnt",count(col("birthDate")).over(window_spec_all)).show()

+------+---------+----------+----------+------+-------------------+-----------+------+----+---------+
|    id|firstName|middleName|  lastName|gender|          birthDate|        ssn|salary|rown|total_cnt|
+------+---------+----------+----------+------+-------------------+-----------+------+----+---------+
|  5093|  Gillian|      Gary|    Durant|     F|1952-01-01 05:00:00|953-85-6672| 71176|   1|      575|
|  5489|    Patsy|     Mandi|    Glancy|     F|1952-01-01 05:00:00|970-75-9602| 66055|   2|      575|
| 27449|    Kayce|     Jonna|  Drysdall|     F|1952-01-01 05:00:00|947-66-4565| 66202|   3|      575|
| 29874|    Dulce|      Irma|     Guild|     F|1952-01-01 05:00:00|907-88-1242| 62662|   4|      575|
| 31237|     Luba|   Mariana|    Cadell|     F|1952-01-01 05:00:00|968-84-2218|131744|   5|      575|
| 51008|Micheline|      Joie| Thomassin|     F|1952-01-01 05:00:00|917-65-1526| 73865|   6|      575|
| 76757|  Tajuana|   Candida|  Gasgarth|     F|1952-01-01 05:00:00|971-93-4959| 71

In [0]:
df_people.withColumn("rown",row_number().over(window_spec)).withColumn("total_cnt",count(col("birthDate")).over(window_spec_all)).printSchema()

root
 |-- id: integer (nullable = true)
 |-- firstName: string (nullable = true)
 |-- middleName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthDate: timestamp (nullable = true)
 |-- ssn: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- rown: integer (nullable = false)
 |-- total_cnt: long (nullable = false)



In [0]:
df_people.withColumn("rown",row_number().over(window_spec)).\
            withColumn("total_cnt",count(col("birthDate")).over(window_spec_all)).\
                withColumn("added_seconds",round((86400/col("total_cnt"))*col("rown"))).\
                    withColumn("new_date",timestamp_add("SECOND",col("added_seconds"),col("birthDate"))).show()

+------+---------+----------+----------+------+-------------------+-----------+------+----+---------+-------------+-------------------+
|    id|firstName|middleName|  lastName|gender|          birthDate|        ssn|salary|rown|total_cnt|added_seconds|           new_date|
+------+---------+----------+----------+------+-------------------+-----------+------+----+---------+-------------+-------------------+
|  5093|  Gillian|      Gary|    Durant|     F|1952-01-01 05:00:00|953-85-6672| 71176|   1|      575|        150.0|1952-01-01 05:02:30|
|  5489|    Patsy|     Mandi|    Glancy|     F|1952-01-01 05:00:00|970-75-9602| 66055|   2|      575|        301.0|1952-01-01 05:05:01|
| 27449|    Kayce|     Jonna|  Drysdall|     F|1952-01-01 05:00:00|947-66-4565| 66202|   3|      575|        451.0|1952-01-01 05:07:31|
| 29874|    Dulce|      Irma|     Guild|     F|1952-01-01 05:00:00|907-88-1242| 62662|   4|      575|        601.0|1952-01-01 05:10:01|
| 31237|     Luba|   Mariana|    Cadell|     F|1

In [0]:
df_people=df_people.withColumn("rown",row_number().over(window_spec)).\
            withColumn("total_cnt",count(col("birthDate")).over(window_spec_all)).\
                withColumn("added_seconds",round((86400/col("total_cnt"))*col("rown"))).\
                    withColumn("new_date",timestamp_add("SECOND",col("added_seconds"),col("birthDate"))).\
                        select("id","firstName","lastName","birthDate","new_date")

In [0]:
df_people.show()

+------+---------+----------+-------------------+-------------------+
|    id|firstName|  lastName|          birthDate|           new_date|
+------+---------+----------+-------------------+-------------------+
|  5093|  Gillian|    Durant|1952-01-01 05:00:00|1952-01-01 05:02:30|
|  5489|    Patsy|    Glancy|1952-01-01 05:00:00|1952-01-01 05:05:01|
| 27449|    Kayce|  Drysdall|1952-01-01 05:00:00|1952-01-01 05:07:31|
| 29874|    Dulce|     Guild|1952-01-01 05:00:00|1952-01-01 05:10:01|
| 31237|     Luba|    Cadell|1952-01-01 05:00:00|1952-01-01 05:12:31|
| 51008|Micheline| Thomassin|1952-01-01 05:00:00|1952-01-01 05:15:02|
| 76757|  Tajuana|  Gasgarth|1952-01-01 05:00:00|1952-01-01 05:17:32|
|134072|     Lois|     Rolse|1952-01-01 05:00:00|1952-01-01 05:20:02|
|141407|   Leanne|    Currum|1952-01-01 05:00:00|1952-01-01 05:22:32|
|151143|    Hilma|     Coale|1952-01-01 05:00:00|1952-01-01 05:25:03|
|199327| Filomena|  Prandini|1952-01-01 05:00:00|1952-01-01 05:27:33|
|231179|     Lise|  

In [0]:
from pyspark.sql.functions import to_date

In [0]:
df_people.where((col("birthDate")> to_date(lit("1952-01-02"))) & (col("birthDate") < to_date(lit("1952-01-03")))).show()

+------+---------+-----------+-------------------+-------------------+
|    id|firstName|   lastName|          birthDate|           new_date|
+------+---------+-----------+-------------------+-------------------+
|   186|    Loise|       Ible|1952-01-02 05:00:00|1952-01-02 05:02:39|
|  7290|    Velva|    Dewdney|1952-01-02 05:00:00|1952-01-02 05:05:17|
|  7501|  Teodora| Sowerbutts|1952-01-02 05:00:00|1952-01-02 05:07:56|
|  9494|   Odessa|Matushevitz|1952-01-02 05:00:00|1952-01-02 05:10:34|
| 10114|   Lezlie|   Stienham|1952-01-02 05:00:00|1952-01-02 05:13:13|
| 25823|    Dulce|    Daniaud|1952-01-02 05:00:00|1952-01-02 05:15:51|
| 42686|   Chante| Matzkaitis|1952-01-02 05:00:00|1952-01-02 05:18:30|
| 45190| Stephine|       Dyos|1952-01-02 05:00:00|1952-01-02 05:21:08|
| 80559|    Myrle|      Maine|1952-01-02 05:00:00|1952-01-02 05:23:47|
|110327|    Ilene|   MacPaike|1952-01-02 05:00:00|1952-01-02 05:26:25|
|122161| Lauretta|  Dominighi|1952-01-02 05:00:00|1952-01-02 05:29:04|
|13557

In [0]:
display(df_people.where((col("birthDate")> to_date(lit("1952-01-02"))) & (col("birthDate") < to_date(lit("1952-01-03")))))

id,firstName,lastName,birthDate,new_date
186,Loise,Ible,1952-01-02T05:00:00.000Z,1952-01-02T05:02:39.000Z
7290,Velva,Dewdney,1952-01-02T05:00:00.000Z,1952-01-02T05:05:17.000Z
7501,Teodora,Sowerbutts,1952-01-02T05:00:00.000Z,1952-01-02T05:07:56.000Z
9494,Odessa,Matushevitz,1952-01-02T05:00:00.000Z,1952-01-02T05:10:34.000Z
10114,Lezlie,Stienham,1952-01-02T05:00:00.000Z,1952-01-02T05:13:13.000Z
25823,Dulce,Daniaud,1952-01-02T05:00:00.000Z,1952-01-02T05:15:51.000Z
42686,Chante,Matzkaitis,1952-01-02T05:00:00.000Z,1952-01-02T05:18:30.000Z
45190,Stephine,Dyos,1952-01-02T05:00:00.000Z,1952-01-02T05:21:08.000Z
80559,Myrle,Maine,1952-01-02T05:00:00.000Z,1952-01-02T05:23:47.000Z
110327,Ilene,MacPaike,1952-01-02T05:00:00.000Z,1952-01-02T05:26:25.000Z


In [0]:
df_people.where((col("birthDate")> to_date(lit("1952-01-03"))) & (col("birthDate") < to_date(lit("1952-01-04")))).count()

563

In [0]:
df_people_count_datewise=df_people.groupby("birthDate").count().orderBy("birthDate").show(5)

+-------------------+-----+
|          birthDate|count|
+-------------------+-----+
|1951-12-31 05:00:00|  112|
|1952-01-01 05:00:00|  575|
|1952-01-02 05:00:00|  545|
|1952-01-03 05:00:00|  563|
|1952-01-04 05:00:00|  576|
+-------------------+-----+
only showing top 5 rows


In [0]:
# check any lag
from pyspark.sql.window import Window
window_spec_only_orderby=Window.orderBy("birthDate")
window_spec_only_orderbyPartitionby=Window.partitionBy("birthDate").orderBy("id")
window_spec_only_orderby

WindowSpec(OrderBy(birthDate ASC NULLS FIRST))

In [0]:
from pyspark.sql.functions import lag

In [0]:
df_people.withColumn("Date_lag",lag("birthdate",1).over(window_spec_only_orderby)).withColumn("Date_lag_with_partion",lag("birthDate",1).over(window_spec_only_orderbyPartitionby)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+---------+---------+-------------------+-------------------+-------------------+---------------------+
|     id|firstName| lastName|          birthDate|           new_date|           Date_lag|Date_lag_with_partion|
+-------+---------+---------+-------------------+-------------------+-------------------+---------------------+
|  49562|    Noemi|   Patise|1951-12-31 05:00:00|1951-12-31 05:12:51|               NULL|                 NULL|
|  61274|  Basilia| Benedick|1951-12-31 05:00:00|1951-12-31 05:25:43|1951-12-31 05:00:00|  1951-12-31 05:00:00|
| 181410|  Elinore|  Cuskery|1951-12-31 05:00:00|1951-12-31 05:38:34|1951-12-31 05:00:00|  1951-12-31 05:00:00|
| 246786|   Louise|  Jeandin|1951-12-31 05:00:00|1951-12-31 05:51:26|1951-12-31 05:00:00|  1951-12-31 05:00:00|
| 445068|Willodean|   Zarfat|1951-12-31 05:00:00|1951-12-31 06:04:17|1951-12-31 05:00:00|  1951-12-31 05:00:00|
| 506248|  Jerrica|McCrackem|1951-12-31 05:00:00|1951-12-31 06:17:09|1951-12-31 05:00:00|  1951-12-31 05

In [0]:
df_people.withColumn("Date_lag",lag("birthdate",1).over(window_spec_only_orderby)).\
         withColumn("Date_lag_with_partion",lag("birthDate",1).over(window_spec_only_orderbyPartitionby)).\
         where(to_date("birthdate") > '1952-01-01').show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+------+---------+-----------+-------------------+-------------------+-------------------+---------------------+
|    id|firstName|   lastName|          birthDate|           new_date|           Date_lag|Date_lag_with_partion|
+------+---------+-----------+-------------------+-------------------+-------------------+---------------------+
|   186|    Loise|       Ible|1952-01-02 05:00:00|1952-01-02 05:02:39|1952-01-01 05:00:00|                 NULL|
|  7290|    Velva|    Dewdney|1952-01-02 05:00:00|1952-01-02 05:05:17|1952-01-02 05:00:00|  1952-01-02 05:00:00|
|  7501|  Teodora| Sowerbutts|1952-01-02 05:00:00|1952-01-02 05:07:56|1952-01-02 05:00:00|  1952-01-02 05:00:00|
|  9494|   Odessa|Matushevitz|1952-01-02 05:00:00|1952-01-02 05:10:34|1952-01-02 05:00:00|  1952-01-02 05:00:00|
| 10114|   Lezlie|   Stienham|1952-01-02 05:00:00|1952-01-02 05:13:13|1952-01-02 05:00:00|  1952-01-02 05:00:00|
| 25823|    Dulce|    Daniaud|1952-01-02 05:00:00|1952-01-02 05:15:51|1952-01-02 05:00:00|  1952

In [0]:
from pyspark.sql.functions import lag,dense_rank

In [0]:
dr_window_spec= Window.orderBy("birthDate")
df_people.withColumn("DR_BDATE",dense_rank().over(dr_window_spec)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------+---------+---------+-------------------+-------------------+--------+
|     id|firstName| lastName|          birthDate|           new_date|DR_BDATE|
+-------+---------+---------+-------------------+-------------------+--------+
| 246786|   Louise|  Jeandin|1951-12-31 05:00:00|1951-12-31 05:51:26|       1|
| 731890|     Tory|     Smye|1951-12-31 05:00:00|1951-12-31 06:30:00|       1|
|1232813| Precious|    Fruen|1951-12-31 05:00:00|1951-12-31 07:08:34|       1|
|1104526|   Melodi|    Plume|1951-12-31 05:00:00|1951-12-31 06:55:43|       1|
|  61274|  Basilia| Benedick|1951-12-31 05:00:00|1951-12-31 05:25:43|       1|
|  49562|    Noemi|   Patise|1951-12-31 05:00:00|1951-12-31 05:12:51|       1|
|1699141|   Elayne|    Knath|1951-12-31 05:00:00|1951-12-31 08:38:34|       1|
|1453976|     Loma|  Fitchet|1951-12-31 05:00:00|1951-12-31 07:47:09|       1|
| 181410|  Elinore|  Cuskery|1951-12-31 05:00:00|1951-12-31 05:38:34|       1|
|1700245|    Manda|  Goodlud|1951-12-31 05:00:00|195

In [0]:
df_people_date_distinct=df_people.withColumn("DR_BDATE",dense_rank().over(dr_window_spec)).select("birthDate","DR_BDATE").distinct()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_people_date_distinct.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+--------+
|          birthDate|DR_BDATE|
+-------------------+--------+
|1951-12-31 05:00:00|       1|
|1952-01-01 05:00:00|       2|
|1952-01-02 05:00:00|       3|
|1952-01-03 05:00:00|       4|
|1952-01-04 05:00:00|       5|
|1952-01-05 05:00:00|       6|
|1952-01-06 05:00:00|       7|
|1952-01-07 05:00:00|       8|
|1952-01-08 05:00:00|       9|
|1952-01-09 05:00:00|      10|
|1952-01-10 05:00:00|      11|
|1952-01-11 05:00:00|      12|
|1952-01-12 05:00:00|      13|
|1952-01-13 05:00:00|      14|
|1952-01-14 05:00:00|      15|
|1952-01-15 05:00:00|      16|
|1952-01-16 05:00:00|      17|
|1952-01-17 05:00:00|      18|
|1952-01-18 05:00:00|      19|
|1952-01-19 05:00:00|      20|
+-------------------+--------+
only showing top 20 rows


In [0]:
df_people_date_distinct.withColumn("prev_date",lag("birthDate",1).over(dr_window_spec)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+--------+-------------------+
|          birthDate|DR_BDATE|          prev_date|
+-------------------+--------+-------------------+
|1951-12-31 05:00:00|       1|               NULL|
|1952-01-01 05:00:00|       2|1951-12-31 05:00:00|
|1952-01-02 05:00:00|       3|1952-01-01 05:00:00|
|1952-01-03 05:00:00|       4|1952-01-02 05:00:00|
|1952-01-04 05:00:00|       5|1952-01-03 05:00:00|
|1952-01-05 05:00:00|       6|1952-01-04 05:00:00|
|1952-01-06 05:00:00|       7|1952-01-05 05:00:00|
|1952-01-07 05:00:00|       8|1952-01-06 05:00:00|
|1952-01-08 05:00:00|       9|1952-01-07 05:00:00|
|1952-01-09 05:00:00|      10|1952-01-08 05:00:00|
|1952-01-10 05:00:00|      11|1952-01-09 05:00:00|
|1952-01-11 05:00:00|      12|1952-01-10 05:00:00|
|1952-01-12 05:00:00|      13|1952-01-11 05:00:00|
|1952-01-13 05:00:00|      14|1952-01-12 05:00:00|
|1952-01-14 05:00:00|      15|1952-01-13 05:00:00|
|1952-01-15 05:00:00|      16|1952-01-14 05:00:00|
|1952-01-16 05:00:00|      17|1

In [0]:
from pyspark.sql.functions import date_diff


In [0]:
df_people_date_distinct.withColumn("prev_date",lag("birthDate",1).over(dr_window_spec)).withColumn("date_difference",date_diff("birthDate","prev_date")).where(col("date_difference") == 1).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+--------+-------------------+---------------+
|          birthDate|DR_BDATE|          prev_date|date_difference|
+-------------------+--------+-------------------+---------------+
|1952-01-01 05:00:00|       2|1951-12-31 05:00:00|              1|
|1952-01-02 05:00:00|       3|1952-01-01 05:00:00|              1|
|1952-01-03 05:00:00|       4|1952-01-02 05:00:00|              1|
|1952-01-04 05:00:00|       5|1952-01-03 05:00:00|              1|
|1952-01-05 05:00:00|       6|1952-01-04 05:00:00|              1|
|1952-01-06 05:00:00|       7|1952-01-05 05:00:00|              1|
|1952-01-07 05:00:00|       8|1952-01-06 05:00:00|              1|
|1952-01-08 05:00:00|       9|1952-01-07 05:00:00|              1|
|1952-01-09 05:00:00|      10|1952-01-08 05:00:00|              1|
|1952-01-10 05:00:00|      11|1952-01-09 05:00:00|              1|
|1952-01-11 05:00:00|      12|1952-01-10 05:00:00|              1|
|1952-01-12 05:00:00|      13|1952-01-11 05:00:00|            

In [0]:
df_people_date_distinct=df_people_date_distinct.withColumn("prev_date",lag("birthDate",1).over(dr_window_spec))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
df_people.join(df_people_date_distinct,on="birthDate",how="left").orderBy("DR_BDATE").where(to_date("birthDate") =='1952-01-02').show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-------------------+------+---------+-----------+-------------------+--------+-------------------+
|          birthDate|    id|firstName|   lastName|           new_date|DR_BDATE|          prev_date|
+-------------------+------+---------+-----------+-------------------+--------+-------------------+
|1952-01-02 05:00:00|235720|  Latasha|  McFarlane|1952-01-02 05:52:51|       3|1952-01-01 05:00:00|
|1952-01-02 05:00:00|179339|     Maud|       Macy|1952-01-02 05:36:59|       3|1952-01-01 05:00:00|
|1952-01-02 05:00:00|196754|Hortencia|      Arnot|1952-01-02 05:44:55|       3|1952-01-01 05:00:00|
|1952-01-02 05:00:00|229579|   Melina|   Rentalll|1952-01-02 05:50:12|       3|1952-01-01 05:00:00|
|1952-01-02 05:00:00|122161| Lauretta|  Dominighi|1952-01-02 05:29:04|       3|1952-01-01 05:00:00|
|1952-01-02 05:00:00|157941|  Scarlet|   Peperell|1952-01-02 05:34:21|       3|1952-01-01 05:00:00|
|1952-01-02 05:00:00|110327|    Ilene|   MacPaike|1952-01-02 05:26:25|       3|1952-01-01 05:00:00|
